In [1]:
from settings import Setting
from data_manager import DataManager
from model_factory import ModelFactory
from trainer import Trainer
import visualizer

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import copy

class SimpleFocalLoss(nn.Module):
    def __init__(self, gamma=2.0, label_smoothing = 0.1):
        super(SimpleFocalLoss, self).__init__()
        self.gamma = gamma # 감마 값이 클수록 어려운 문제에 더 집착합니다 (기본 2.0 추천)
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        # 1. 일반적인 CrossEntropy를 계산합니다. (reduction='none'으로 각 샘플별 손실 유지)
        ce_loss = F.cross_entropy(inputs, targets, reduction='none',
                                  label_smoothing=self.label_smoothing)
        
        # 2. 모델이 정답을 맞출 확률(pt)을 구합니다.
        pt = torch.exp(-ce_loss) 
        
        # 3. 어려운 문제일수록(pt가 낮을수록) 손실값을 증폭시킵니다.
        focal_loss = ((1 - pt) ** self.gamma * ce_loss).mean()
        
        return focal_loss


class EarlyStopping:
    def __init__(self, patience=7, delta=0.001):
        self.patience = patience  # 개선되지 않을 때 참아줄 에포크 횟수
        self.delta = delta     # 개선되었다고 판단할 최소 변화량
        self.counter = 0          # 개선 안 된 횟수 카운터
        self.best_loss = None     # 가장 좋았던(낮았던) 오차
        self.early_stop = False   # 종료 여부 플래그
        self.is_best_model = True

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.is_best_model = True
        elif val_loss > self.best_loss - self.delta:
            self.counter += 1
            self.is_best_model = False
            print(f"조기 종료 카운트: {self.counter} / {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.is_best_model = True
            self.counter = 0 # 개선되면 카운터 초기화

setting = Setting() 
data_manager = DataManager(setting)

print(f"이미지: {data_manager._data_size} 장, 클래스 개수: {data_manager.num_classes} 개")
print(f"드롭아웃 비율: {setting.DROPOUT_RATE}")
print(f"가중치 감쇠 정도: {setting.WEIGHT_DECAY}")
print(f"데이터 증강: {'적용' if setting.CAN_USE_AUGMENTATION else '미적용'}")
print(f"학습률 스케줄러: {'적용' if setting.CAN_USE_SCHEDULER else '미적용'}")

for name in setting.MODEL_NAME_LIST:
    print(f"\n--- {name} 모델 실험 시작 ---")
    
    # 모델과 전처리 규칙을 가져옴
    model, train_transform, val_transform = ModelFactory.create_model_and_transforms(name, 
                                            data_manager.num_classes, setting.DROPOUT_RATE, 
                                            setting.CAN_USE_AUGMENTATION)
    
    # 가져온 전처리 규칙으로 데이터를 모델 학습용 및 평가용으로 가공한 객체인 로더를 가져옴
    train_loader, val_loader = data_manager.get_loaders(train_transform, val_transform)

    # 손실 함수는 획득한 결과와 실제 값 사이의 틀린 정도를 측정하는 함수
    # 학습 중에 이 값을 최소화하려고 하며, 예측과 정답을 비교해 손실을 계산
    # label_smoothing은 과적합을 막기 위해 데이터 정답 확신도 일부를 다른 클래스에 배분하는 비율
    # 모델 매개변수 최적화하기 - 파이토치 한국어 튜토리얼
    # https://tutorials.pytorch.kr/beginner/basics/optimization_tutorial.html
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    # criterion = SimpleFocalLoss(gamma=2.0, label_smoothing=0.1)

    # 옵티마이저는 손실 함수의 최저점을 찾아주는 탐색기

    # 가중치 재학습 여부를 False로 한 것들은 옵티마이저에 넣을 필요가 없음
    # 따라서 실제로 학습할 파라미터들만 모으는 리스트를 따로 생성
    # 이미 모델 생성 과정에서 마지막 레이어만 True가 되었으니 그것만 들어갈 예정
    param_list = [] 

    for param in model.parameters():
        if param.requires_grad == True:
            param_list.append(param) 
    
    # 컴퓨터 비전을 위한 전이 학습 튜토리얼에서는 옵티마이저로 SGD 사용
    # SGD 옵티마이저에 재학습 가능한 파라미터만 있는 리스트, 학습률, 모멘텀(관성) 전달해 객체 생성
    # optimizer = optim.SGD(param_list, lr=setting.LEARNING_RATE, momentum=setting.OPTIM_MOMENTUM,
    #                       weight_decay=setting.WEIGHT_DECAY)
    
    # 옵티마이저로 Adam을 쓴 버전, SGD와 바꿔가며 테스트
    optimizer = optim.Adam(param_list, lr=setting.LEARNING_RATE, weight_decay=setting.WEIGHT_DECAY)

    # optimizer = optim.AdamW(param_list, lr=setting.LEARNING_RATE, weight_decay=setting.WEIGHT_DECAY)

    if setting.CAN_USE_SCHEDULER:
        # 학습률 스케줄러를 이용하면 중간에 옵티마이저 학습률을 자동으로 낮춰줄 수 있음
        # factor는 학습률을 얼마나 줄일지를, patience는 몇 에포크 동안 안 줄어들면 줄일지를 나타냄
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    # 학습용 엔진에 모델, 설정, 손실 함수, 옵티마이저를 전달해 객체 생성
    trainer = Trainer(model, setting, criterion, optimizer)

    # 인공지능 강의 #6 5장 딥러닝과 텐서플로를 참고
    # 텐서플로에서는 model.fit() 메서드가 학습 도중에 발생한 정보를 hist 객체에 저장해 둠
    # hist.history['accuracy']처럼 쓰기만 해도 바로 시각화에 쓸 수 있음
    # 그런데 파이토치에는 그런 기능이 없으므로 따로 딕셔너리 만들어 기록할 필요가 있음
    history = {
        'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []
    }

    # 루프 시작 전 객체 생성
    early_stopping = EarlyStopping(patience=setting.EARLY_STOP_PATIENCE) # n번 연속으로 안 좋아지면 멈춤

    best_val_loss = float('inf')
    best_model = copy.deepcopy(model.state_dict())

    # 정해진 횟수만큼 에포크 반복
    for epoch in range(setting.EPOCHS):
        print(f"\n[Epoch {epoch+1}/{setting.EPOCHS} 시작]")

        # 먼저 전체 학습 데이터셋에 대해 한 바퀴 훈련한 뒤 전체 평균 오차와 정확도 출력
        train_loss, train_acc = trainer.train_epoch(train_loader)
        print(f"[Train] Loss: {train_loss:.4f} | Acc: {train_acc * 100:.2f}%")

        # 그 다음 가중치 수정 없이 현재 모델 평가 후 평균 오차와 정확도 출력
        val_loss, val_acc = trainer.evaluate(val_loader)
        print(f"[ Val ] Loss: {val_loss:.4f} | Acc: {val_acc * 100:.2f}%")

        if setting.CAN_USE_SCHEDULER:
            # 검증 오차를 기준으로 학습률 조정하겠다는 뜻
            scheduler.step(val_loss)

        if setting.CAN_DRAW_PLOT:
            history['train_loss'].append(train_loss)
            # 정확도 객체는 파이토치의 텐서 객체로, 그대로 들어가면 충돌 또는 메모리 점유 위험 가능성 있음
            # 따라서 item()을 붙여 순수한 숫자 데이터로 바꿔 넣는 것이 안전
            history['train_acc'].append(train_acc.item())
            history['val_loss'].append(val_loss)
            history['val_acc'].append(val_acc.item())

        # [추가] 조기 종료 로직 실행
        early_stopping(val_loss)
        if early_stopping.early_stop:
            print("학습이 더 이상 개선되지 않아 조기에 종료합니다.")
            break  # 에포크 반복문 탈출
        elif early_stopping.is_best_model:
            best_model = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_model)

    # --- [ 심화 오답 분석: 혼동 행렬 ] ---
    model.eval()
    class_names = data_manager._classes

    # '실제 클래스'가 '예측 클래스'로 분류된 횟수를 저장할 딕셔너리
    # 예: confusion_matrix['crying']['sad'] = 10 이면 우는 표정을 슬픈 표정으로 10번 착각함
    confusion_matrix = {actual: {pred: 0 for pred in class_names} for actual in class_names}

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(setting.DEVICE)
            labels = labels.to(setting.DEVICE)
            
            outputs = model(inputs)
            _, predictions = torch.max(outputs, 1)
            
            for l, p in zip(labels, predictions):
                actual_name = class_names[l]
                pred_name = class_names[p]
                confusion_matrix[actual_name][pred_name] += 1

    # 전체 맞춘 개수와 전체 데이터 개수 계산
    total_correct = sum(confusion_matrix[classname][classname] for classname in class_names)
    total_samples = sum(sum(confusion_matrix[actual].values()) for actual in class_names)
    best_accuracy = (total_correct / total_samples) * 100

    # 결과 출력
    print("\n" + "="*50)
    print(" [ 혼동 행렬 (Confusion Matrix) 분석 ]")
    print(f" ▶ 복구된 최고 모델의 최종 Val 정확도: {best_accuracy:.2f}%")
    print(" 세로축: 실제 정답 / 가로축: 모델의 예측")
    print("="*50)

    # 헤더 출력 (예측 클래스 이름들)
    header = " " * 8 + " | ".join([f"{n[:5]:>5}" for n in class_names])
    print(header)
    print("-" * len(header))

    for actual in class_names:
        row = f"{actual[:5]:>5} | "
        row += " | ".join([f"{confusion_matrix[actual][pred]:>5}" for pred in class_names])
        
        # 본인 클래스를 맞춘 숫자는 [ ]로 표시하여 가독성 높임
        # (행렬의 대각선 성분이 높을수록 좋은 모델)
        print(row)

    print("\n" + "="*50)
    print(" [ 주요 오답 패턴 분석 ]")
    print("="*50)

    for actual in class_names:
        # 해당 클래스에서 틀린 것들만 추출
        errors = {p: count for p, count in confusion_matrix[actual].items() if p != actual and count > 0}
        if errors:
            # 가장 많이 착각한 순서대로 정렬
            sorted_errors = sorted(errors.items(), key=lambda x: x[1], reverse=True)
            top_mistake, count = sorted_errors[0]
            print(f"'{actual}' 표정을 가장 많이 착각한 대상: '{top_mistake}' ({count}회)")
            
    if setting.CAN_DRAW_PLOT:
        visualizer.draw_plot(history)

    print(f"\n--- {name} 모델 실험 끝 ---")

엔비디아 GPU cuda 사용
이미지: 296 장, 클래스 개수: 7 개
드롭아웃 비율: 0.5
가중치 감쇠 정도: 0.05
데이터 증강: 적용
학습률 스케줄러: 적용

--- resnet50 모델 실험 시작 ---

[Epoch 1/100 시작]
[Train] Loss: 1.9119 | Acc: 19.26%
[ Val ] Loss: 1.9270 | Acc: 25.79%

[Epoch 2/100 시작]
[Train] Loss: 1.7638 | Acc: 38.85%
[ Val ] Loss: 1.8429 | Acc: 30.19%

[Epoch 3/100 시작]
[Train] Loss: 1.5414 | Acc: 49.66%
[ Val ] Loss: 2.1487 | Acc: 30.82%
조기 종료 카운트: 1 / 30

[Epoch 4/100 시작]
[Train] Loss: 1.3407 | Acc: 56.42%
[ Val ] Loss: 1.4072 | Acc: 50.94%

[Epoch 5/100 시작]
[Train] Loss: 1.1670 | Acc: 66.22%
[ Val ] Loss: 1.2184 | Acc: 66.67%

[Epoch 6/100 시작]
[Train] Loss: 1.0057 | Acc: 75.68%
[ Val ] Loss: 1.2710 | Acc: 62.26%
조기 종료 카운트: 1 / 30

[Epoch 7/100 시작]
[Train] Loss: 1.0584 | Acc: 73.31%
[ Val ] Loss: 1.3745 | Acc: 66.04%
조기 종료 카운트: 2 / 30

[Epoch 8/100 시작]
[Train] Loss: 1.0821 | Acc: 75.68%
[ Val ] Loss: 1.4738 | Acc: 57.23%
조기 종료 카운트: 3 / 30

[Epoch 9/100 시작]
[Train] Loss: 1.0191 | Acc: 77.03%
[ Val ] Loss: 1.2696 | Acc: 65.41%
조기 종료 카운트: 4 / 